# Grid-Aware MADRL Training Notebook

This notebook trains the Gymnasium-based grid environment with hybrid cooperative rewards, prefers CUDA on the local RTX 5080, and only plots final post-training results.

In [ ]:
from pathlib import Path
import sys
import warnings

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'configs').exists():
    project_root = project_root.parent
if not (project_root / 'configs').exists():
    raise RuntimeError('Could not locate the project root.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

warnings.filterwarnings('ignore', message='The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*')
project_root

In [ ]:
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from scripts.analysis import estimate_static_grid_sensitivity
from configs import compose_experiment_config
from configs.profiles import apply_grid_profile
from controllers import MADRLController, ZeroController
from scripts.builder import build_env
from scripts.evaluate import evaluate_controller
from scripts.plots.grid_plots import plot_publication_day_summary
from scripts.plots.reward_plots import plot_reward_decomposition
from scripts.utils.experiment_notebook_utils import build_runner, get_madrl_checkpoint_root, summarize_cfg
from scripts.utils.torch_runtime import configure_torch_runtime, describe_device

In [ ]:
algorithm = 'MADDPG'
train_episodes = 64
sampler_benchmark_episodes = 2
reward_plot_window = 10
n_summary_eval_episodes = 3
seed = 7
runtime_mode = 'performance'
device_request = 'cuda'
require_cuda = True

cfg = compose_experiment_config(
    profile='debug',
    algorithm=algorithm,
    model_family='mlp',
    reward_type='composite',
    observation_profile='simbench',
    forecast_type='perfect',
    vec_env_type='subproc',
    data_dir=project_root / 'data',
    device=device_request,
    runtime_mode=runtime_mode,
    seed=seed,
    require_cuda=require_cuda,
)
apply_grid_profile(cfg, 'rural1_phase1')

cfg.train.train_episodes = train_episodes
cfg.grid.w_line_pen = float(getattr(cfg.grid, 'w_line_pen', cfg.grid.w_l_pen))
cfg.grid.w_trafo_pen = float(getattr(cfg.grid, 'w_trafo_pen', cfg.grid.w_l_pen))
cfg.train.num_envs = 8
cfg.train.vec_env_type = 'subproc'
cfg.train.batch_size = 4096
cfg.train.buffer_size = 100000
cfg.train.update_interval = 1
cfg.train.updates_per_step = 1
cfg.train.use_noise_decay = True
cfg.train.noise_std_init = 0.35
cfg.train.noise_std_min = 0.05
cfg.train.max_train_steps = None
cfg.train.noise_decay_steps = cfg.train.train_episodes * cfg.env.episode_limit

runtime_state = configure_torch_runtime(cfg, device=device_request, seed=seed, require_cuda=require_cuda)
summary = summarize_cfg(cfg)
summary['device_info'] = describe_device(runtime_state)
summary['torch_cuda_name'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
summary['grid_penalty_weights'] = {
    'voltage': float(cfg.grid.w_v_pen),
    'line': float(cfg.grid.w_line_pen),
    'transformer': float(cfg.grid.w_trafo_pen),
}
summary

In [ ]:
def run_perf_probe(base_cfg, *, num_envs: int, vec_env_type: str, episodes: int, label: str) -> dict:
    probe_cfg = deepcopy(base_cfg)
    probe_cfg.train.train_episodes = episodes
    probe_cfg.train.max_train_steps = None
    probe_cfg.train.num_envs = num_envs
    probe_cfg.train.vec_env_type = vec_env_type
    probe_cfg.train.batch_size = 128
    probe_cfg.train.buffer_size = 4096
    probe_cfg.train.updates_per_step = 1
    probe_cfg.train.update_interval = 1
    probe_runner = build_runner(probe_cfg, seed=seed, env_name=f'GridPerfProbe_{label}', number=1)
    try:
        probe_runner.run()
        perf = dict(probe_runner.perf_summary)
        perf['label'] = label
        return perf
    finally:
        probe_runner.close()

perf_baseline = run_perf_probe(cfg, num_envs=1, vec_env_type='dummy', episodes=sampler_benchmark_episodes, label='1x_dummy')
perf_parallel = run_perf_probe(cfg, num_envs=8, vec_env_type='subproc', episodes=sampler_benchmark_episodes, label='8x_subproc')
perf_compare = pd.DataFrame([
    {
        'label': perf_baseline['label'],
        'steps_per_sec': perf_baseline['steps_per_sec'],
        'avg_action_ms_per_iter': perf_baseline['avg_action_ms_per_iter'],
        'avg_env_ms_per_iter': perf_baseline['avg_env_ms_per_iter'],
        'avg_update_ms_per_call': perf_baseline['avg_update_ms_per_call'],
    },
    {
        'label': perf_parallel['label'],
        'steps_per_sec': perf_parallel['steps_per_sec'],
        'avg_action_ms_per_iter': perf_parallel['avg_action_ms_per_iter'],
        'avg_env_ms_per_iter': perf_parallel['avg_env_ms_per_iter'],
        'avg_update_ms_per_call': perf_parallel['avg_update_ms_per_call'],
    },
])
perf_compare

runner = build_runner(cfg, seed=seed, env_name='GridTrainGymnasiumHybridReward', number=1)

_, reset_info = runner.env_evaluate.reset(episode_idx=0)
print('Gymnasium reset info keys:', sorted(reset_info.keys()))
print('Battery capacities (kWh):', np.round(reset_info['battery_capacity_kwh'], 3))
print('Power limits (kW):', np.round(reset_info['p_max'], 3))
print('Selected CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

episodes_completed = runner.run()
print(f'Training finished: {episodes_completed} episodes')
runner.perf_summary

In [ ]:
def summarize_eval_results(label: str, results: dict) -> dict:
    grid_histories = results.get('grid_histories', [])
    total_steps = sum(len(gh['pf_converged']) for gh in grid_histories)
    converged_steps = sum(sum(bool(flag) for flag in gh['pf_converged']) for gh in grid_histories)
    total_v = sum(sum(gh['n_v_violations']) for gh in grid_histories)
    total_l = sum(sum(gh['n_line_violations']) for gh in grid_histories)
    total_t = sum(sum(gh.get('n_t_violations', [])) for gh in grid_histories)
    max_line = max((float(np.max(arr)) for gh in grid_histories for arr in gh['line_loading_pct'] if len(arr)), default=0.0)
    max_trafo = max((float(np.max(arr)) for gh in grid_histories for arr in gh['trafo_loading_pct'] if len(arr)), default=0.0)
    min_vm = min((float(np.min(np.asarray(gh['agent_vm_pu'], dtype=np.float32))) for gh in grid_histories), default=np.nan)
    max_vm = max((float(np.max(np.asarray(gh['agent_vm_pu'], dtype=np.float32))) for gh in grid_histories), default=np.nan)
    summary = {
        'label': label,
        'mean_episode_reward': float(results['mean_episode_reward']),
        'total_steps': int(total_steps),
        'converged_steps': int(converged_steps),
        'total_voltage_violations': int(total_v),
        'total_line_violations': int(total_l),
        'total_trafo_violations': int(total_t),
        'min_agent_vm_pu': float(min_vm),
        'max_agent_vm_pu': float(max_vm),
        'max_line_loading_pct': float(max_line),
        'max_trafo_loading_pct': float(max_trafo),
    }
    print(summary)
    return summary


def select_typical_days(cfg, env, quantiles=(0.2, 0.5, 0.8)):
    csv_path = cfg.data.data_dir / 'simbench_2016_test.csv'
    df = pd.read_csv(csv_path)
    load_cols = [c for c in df.columns if c.startswith('load')]
    pv_cols = [c for c in df.columns if c.startswith('pv')]
    net = df[load_cols].sum(axis=1) - df[pv_cols].sum(axis=1)

    steps_per_day = int(round(24 / cfg.env.dt))
    days_per_episode = max(1, cfg.env.episode_limit // steps_per_day)
    available_days = env.num_available_episodes * days_per_episode

    daily_mean = net.groupby(np.arange(len(df)) // steps_per_day).mean().iloc[:available_days]
    labels = ['low', 'mid', 'high'][: len(quantiles)]
    selected = []
    used_days = set()
    for label, quantile in zip(labels, quantiles):
        target = daily_mean.quantile(quantile)
        for day_idx in (daily_mean - target).abs().sort_values().index.tolist():
            if int(day_idx) in used_days:
                continue
            used_days.add(int(day_idx))
            selected.append({
                'label': label,
                'day_idx': int(day_idx),
                'episode_idx': int(day_idx // days_per_episode),
                'day_offset': int(day_idx % days_per_episode),
                'daily_mean_net_kw': float(daily_mean.loc[day_idx]),
            })
            break
    return selected, steps_per_day


def slice_grid_history(grid_history: dict, start: int, end: int) -> dict:
    return {
        'agent_vm_pu': [series[start:end] for series in grid_history['agent_vm_pu']],
        'line_loading_pct': grid_history['line_loading_pct'][start:end],
        'trafo_loading_pct': grid_history['trafo_loading_pct'][start:end],
        'line_violation': grid_history.get('line_violation', [])[start:end],
        'trafo_violation': grid_history.get('trafo_violation', [])[start:end],
        'n_v_violations': grid_history['n_v_violations'][start:end],
        'n_l_violations': grid_history['n_l_violations'][start:end],
        'n_line_violations': grid_history.get('n_line_violations', [])[start:end],
        'n_t_violations': grid_history.get('n_t_violations', [])[start:end],
        'n_trafo_violations': grid_history.get('n_trafo_violations', [])[start:end],
        'pf_converged': grid_history['pf_converged'][start:end],
    }


def slice_episode_history(history: dict, start: int, end: int) -> dict:
    sliced = {
        'price': history['price'][start:end],
        'base_net_load': [series[start:end] for series in history.get('base_net_load', [])],
        'e_bat_req': [series[start:end] for series in history['e_bat_req']],
        'e_bat_exec': [series[start:end] for series in history['e_bat_exec']],
        'soc': [series[start : end + 1] for series in history['soc']],
        'r_total_sum': history['r_total_sum'][start:end],
        'r_total_per_agent': [series[start:end] for series in history.get('r_total_per_agent', [])],
    }
    for key, value in history.items():
        if key.endswith('_per_agent'):
            sliced[key] = [series[start:end] for series in value]
    return sliced


def plot_per_agent_reward_components(history: dict, reward_fn, agent_labels=None, title: str = 'Per-Agent Reward Components'):
    metas = list(getattr(reward_fn, 'component_meta', []))
    n_agents = len(history.get('e_bat_exec', []))
    if n_agents == 0:
        raise ValueError('history does not contain per-agent traces.')
    agent_labels = agent_labels or [f'Agent {idx}' for idx in range(n_agents)]
    n_steps = len(history.get('price', []))
    t = np.arange(n_steps)

    fig, axes = plt.subplots(
        n_agents,
        1,
        figsize=(14, max(3.2 * n_agents, 4.0)),
        sharex=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)

    for agent_idx, ax in enumerate(axes):
        reconstructed_total = np.zeros(n_steps, dtype=np.float32)
        for meta in metas:
            key = f'{meta.key}_per_agent'
            if key not in history:
                continue
            series = np.asarray(history[key][agent_idx], dtype=np.float32)
            if series.size != n_steps:
                continue
            reconstructed_total += series
            ax.plot(t, series, label=meta.label, color=meta.color, linewidth=1.5, alpha=0.9)

        total_series = np.asarray(history.get('r_total_per_agent', [[] for _ in range(n_agents)])[agent_idx], dtype=np.float32)
        if total_series.size == n_steps:
            ax.plot(t, total_series, label='Total reward', color='black', linewidth=2.2)
        else:
            ax.plot(t, reconstructed_total, label='Total reward', color='black', linewidth=2.2)

        ax.axhline(0.0, color='0.45', linestyle='--', linewidth=0.9)
        ax.set_title(agent_labels[agent_idx])
        ax.set_ylabel('Reward')
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel('Timestep')
    handles, labels = axes[0].get_legend_handles_labels()
    legend_cols = min(max(len(labels), 1), 3)
    axes[0].legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.35), ncol=legend_cols, fontsize=8)
    fig.suptitle(title)
    return fig, axes


def compute_operating_cost_per_agent(histories: list[dict], dt: float) -> np.ndarray:
    if not histories:
        return np.zeros(0, dtype=np.float32)

    totals = None
    for history in histories:
        price = np.asarray(history['price'], dtype=np.float64)
        base_net_load = np.asarray(history['base_net_load'], dtype=np.float64)
        e_bat_exec = np.asarray(history['e_bat_exec'], dtype=np.float64)
        if base_net_load.size == 0:
            raise ValueError('Episode history is missing base_net_load traces.')

        operating_cost = np.sum((base_net_load + e_bat_exec) * price[np.newaxis, :] * float(dt), axis=1)
        if totals is None:
            totals = np.zeros_like(operating_cost, dtype=np.float64)
        totals += operating_cost

    return totals.astype(np.float32)


def plot_operating_cost_comparison(
    baseline_histories: list[dict],
    controlled_histories: list[dict],
    dt: float,
    agent_labels=None,
    title: str = 'Real Operating Cost Comparison',
) -> pd.DataFrame:
    baseline_cost = compute_operating_cost_per_agent(baseline_histories, dt)
    madrl_cost = compute_operating_cost_per_agent(controlled_histories, dt)
    n_agents = len(baseline_cost)
    agent_labels = agent_labels or [f'Agent {idx}' for idx in range(n_agents)]

    x = np.arange(n_agents)
    width = 0.36
    fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
    ax.bar(x - width / 2, baseline_cost, width=width, color='0.75', label='Baseline (ZeroController)')
    ax.bar(x + width / 2, madrl_cost, width=width, color='tab:blue', label='MADRL final policy')
    ax.set_xticks(x, agent_labels)
    ax.set_ylabel('Operating cost (price-energy units)')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.25)
    ax.legend()

    savings = baseline_cost - madrl_cost
    savings_pct = np.where(np.abs(baseline_cost) > 1e-6, 100.0 * savings / np.abs(baseline_cost), np.nan)
    return pd.DataFrame(
        {
            'agent': agent_labels,
            'baseline_cost': baseline_cost,
            'madrl_cost': madrl_cost,
            'cost_saving': savings,
            'cost_saving_pct': savings_pct,
        }
    ).round(4)


def evaluate_one_episode(cfg, controller, episode_idx: int) -> dict:
    env = build_env(cfg, mode='test')
    try:
        results = evaluate_controller(
            env=env,
            controller=controller,
            n_episodes=1,
            deterministic=True,
            episode_indices=[episode_idx],
            record_history=True,
        )
    finally:
        env.close()
    return results


trained_controller = MADRLController(runner.agent_n, noise_std=0.0)
zero_controller = ZeroController(action_dim_n=[1] * cfg.env.num_agents)


In [ ]:
plot_reward_decomposition(
    history=list(runner.history),
    episode_rewards=runner.episode_rewards,
    reward_fn=runner.env_evaluate.reward_fn,
    title='Training Reward Decomposition',
    window=reward_plot_window,
)

sensitivity_snapshot = estimate_static_grid_sensitivity(cfg, episode_idx=0, step_idx=0, delta_kw=1.0)
sensitivity_summary = pd.DataFrame(
    sensitivity_snapshot['voltage_sensitivity_pu_per_kw'],
    index=[f"Bus {bid}" for bid in cfg.grid.agent_bus_ids],
    columns=[f"Agent@Bus {bid}" for bid in cfg.grid.agent_bus_ids],
)
print('Offline voltage sensitivity snapshot (pu per kW):')
print(sensitivity_summary)

summary_episode_indices = list(range(min(n_summary_eval_episodes, runner.env_evaluate.num_available_episodes)))

summary_env = build_env(cfg, mode='test')
try:
    zero_summary = evaluate_controller(
        env=summary_env,
        controller=zero_controller,
        n_episodes=len(summary_episode_indices),
        deterministic=True,
        episode_indices=summary_episode_indices,
        record_history=True,
    )
finally:
    summary_env.close()

summary_env = build_env(cfg, mode='test')
try:
    trained_summary = evaluate_controller(
        env=summary_env,
        controller=trained_controller,
        n_episodes=len(summary_episode_indices),
        deterministic=True,
        episode_indices=summary_episode_indices,
        record_history=True,
    )
finally:
    summary_env.close()

zero_stats = summarize_eval_results('baseline_zero_controller', zero_summary)
trained_stats = summarize_eval_results('trained_madrl_controller', trained_summary)

print('Power-flow convergence over the trained-controller summary set:', f"{trained_stats['converged_steps']}/{trained_stats['total_steps']}")

In [ ]:
plot_per_agent_reward_components(
    history=trained_summary['histories'][0],
    reward_fn=runner.env_evaluate.reward_fn,
    agent_labels=[f"Bus {bid}" for bid in cfg.grid.agent_bus_ids],
    title=(
        'Per-Agent Reward Components | final MADRL policy | '
        f"test episode index {summary_episode_indices[0]}"
    ),
)

In [ ]:
operating_cost_df = plot_operating_cost_comparison(
    baseline_histories=zero_summary['histories'],
    controlled_histories=trained_summary['histories'],
    dt=cfg.env.dt,
    agent_labels=[f"Bus {bid}" for bid in cfg.grid.agent_bus_ids],
    title='Real Operating Cost Comparison | held-out test summary episodes',
)
operating_cost_df

In [ ]:
typical_days, steps_per_day = select_typical_days(cfg, runner.env_evaluate)
typical_days

In [ ]:
for day_info in typical_days:
    episode_idx = day_info['episode_idx']
    start = day_info['day_offset'] * steps_per_day
    end = start + steps_per_day

    baseline_result = evaluate_one_episode(cfg, zero_controller, episode_idx)
    controlled_result = evaluate_one_episode(cfg, trained_controller, episode_idx)

    baseline_grid = slice_grid_history(baseline_result['grid_histories'][0], start, end)
    controlled_grid = slice_grid_history(controlled_result['grid_histories'][0], start, end)
    controlled_history = slice_episode_history(controlled_result['histories'][0], start, end)

    baseline_vm = np.asarray(baseline_grid['agent_vm_pu'], dtype=np.float32)
    controlled_vm = np.asarray(controlled_grid['agent_vm_pu'], dtype=np.float32)
    baseline_line = max((float(np.max(arr)) for arr in baseline_grid['line_loading_pct'] if len(arr)), default=0.0)
    controlled_line = max((float(np.max(arr)) for arr in controlled_grid['line_loading_pct'] if len(arr)), default=0.0)
    baseline_trafo = max((float(np.max(arr)) for arr in baseline_grid['trafo_loading_pct'] if len(arr)), default=0.0)
    controlled_trafo = max((float(np.max(arr)) for arr in controlled_grid['trafo_loading_pct'] if len(arr)), default=0.0)

    print(
        f"{day_info['label']} day | test episode index={episode_idx} | mean net={day_info['daily_mean_net_kw']:.3f} kW | "
        f"baseline vm=[{baseline_vm.min():.4f}, {baseline_vm.max():.4f}] line={baseline_line:.1f}% trafo={baseline_trafo:.1f}% | "
        f"controlled vm=[{controlled_vm.min():.4f}, {controlled_vm.max():.4f}] line={controlled_line:.1f}% trafo={controlled_trafo:.1f}% | "
        f"controlled converged={sum(controlled_grid['pf_converged'])}/{len(controlled_grid['pf_converged'])}"
    )

    plot_publication_day_summary(
        history=controlled_history,
        baseline_grid_history=baseline_grid,
        controlled_grid_history=controlled_grid,
        title=f"Typical {day_info['label'].title()}-Net-Load Day: Final Trained Policy",
        day_caption=(
            f"Test episode index {episode_idx}, test day offset {day_info['day_offset']} | "
            f"mean net load = {day_info['daily_mean_net_kw']:.2f} kW"
        ),
        v_min=cfg.grid.v_min_pu,
        v_max=cfg.grid.v_max_pu,
        agent_labels=[f"Bus {bid}" for bid in cfg.grid.agent_bus_ids],
    )

In [ ]:
save_dir = get_madrl_checkpoint_root(project_root) / 'MADDPG_Grid_Gymnasium_GPU'
save_dir.mkdir(parents=True, exist_ok=True)
runner.save_model(str(save_dir), episode=episodes_completed)
runner.close()
print(f'Model saved to: {save_dir}')